# 01 - Synthetic Dataset Generation

Generates a synthetic function-calling dataset for fine-tuning Mistral 7B.

**Approach**: Use Claude via the Databricks Foundation Model API to generate 
diverse user queries paired with correct tool call responses across 16 enterprise 
tool schemas and 5 difficulty categories. Validates against tool schemas, and saves train/val/test splits.

**Output**: `train.jsonl`, `val.jsonl`, `test.jsonl`

In [0]:
import sys
import os
import pandas as pd
from sklearn.model_selection import train_test_split

# Add project root to path so we can import src/
# Adjust this path to wherever your repo is cloned/mounted in Databricks
PROJECT_ROOT = "/Workspace/Users/xxx"
sys.path.insert(0, PROJECT_ROOT)

from src.schemas import TOOL_SCHEMAS, TOOL_NAMES, SYSTEM_PROMPT
from src.generation import build_generation_plan, run_generation
from src.validation import parse_model_response
from src.utils import (
    to_training_format, deduplicate_examples,
    save_jsonl, load_jsonl, spot_check, dataset_stats
)

In [0]:
from openai import OpenAI

DATABRICKS_HOST = (
    dbutils.notebook.entry_point.getDbutils()
    .notebook().getContext().apiUrl().getOrElse(None)
)
DATABRICKS_TOKEN = (
    dbutils.notebook.entry_point.getDbutils()
    .notebook().getContext().apiToken().getOrElse(None)
)

client = OpenAI(
    api_key=DATABRICKS_TOKEN,
    base_url=f"{DATABRICKS_HOST}/serving-endpoints"
)

CLAUDE_MODEL = "databricks-claude-sonnet-4-6"  # <-- UPDATE to match your endpoint

def call_claude(prompt: str) -> str:
    """Call Claude Opus via Databricks."""
    response = client.chat.completions.create(
        model=CLAUDE_MODEL,
        messages=[{"role": "user", "content": prompt}],
        max_tokens=4096,
        temperature=0.8,
    )
    return response.choices[0].message.content

# Quick test
print(call_claude("Say 'hello' and nothing else."))

In [0]:
plan = build_generation_plan()
print(f"Total generation tasks: {len(plan)}")

summary = pd.DataFrame([
    {"category": t.category, "expected": t.expected_count} for t in plan
])
print(summary.groupby("category")["expected"].sum())
print(f"\nTotal expected: {summary['expected'].sum()}")

In [0]:
raw_examples, gen_log = run_generation(
    tasks=plan,
    llm_call_fn=call_claude,
    max_retries=2,
    delay_between_calls=1.5,
)

print(f"\n✅ Generated {len(raw_examples)} valid examples")

In [0]:
print(f"Successful: {len(gen_log[gen_log['status'] == 'success'])}")
print(f"Failed: {len(gen_log[gen_log['status'] != 'success'])}")
print(f"\nValid by category:\n{gen_log.groupby('category')['valid'].sum()}")
print(f"\nInvalid by category:\n{gen_log.groupby('category')['invalid'].sum()}")

failed = gen_log[gen_log["status"] != "success"]
if len(failed) > 0:
    print(f"\n⚠️ Failed tasks:\n{failed[['task_id', 'status']]}")

In [0]:
training_examples = [to_training_format(ex) for ex in raw_examples]
training_examples = deduplicate_examples(training_examples)

In [0]:
# Extract categories for stratification
categories = [ex.get("category", ex.get("_category", "unknown")) for ex in training_examples]

# First split: 80% train, 20% temp (val + test)
train_set, temp_set, train_cats, temp_cats = train_test_split(
    training_examples, categories, test_size=0.2, stratify=categories, random_state=42
)

# Second split: split temp 50/50 into val (10%) and test (10%)
val_set, test_set = train_test_split(
    temp_set, test_size=0.5, stratify=temp_cats, random_state=42
)

print(f"Train: {len(train_set)} | Val: {len(val_set)} | Test: {len(test_set)}")

In [0]:
for cat in ["simple", "complex", "multi_tool", "ambiguous", "no_tool"]:
    print(f"\n{'#'*70}")
    print(f"# {cat.upper()}")
    print(f"{'#'*70}")
    spot_check(training_examples, n=3, category=cat)

In [0]:
OUTPUT_DIR = f"{PROJECT_ROOT}/data"
os.makedirs(OUTPUT_DIR, exist_ok=True)

save_jsonl(train_set, f"{OUTPUT_DIR}/train.jsonl", strip_category=True)
save_jsonl(val_set, f"{OUTPUT_DIR}/val.jsonl", strip_category=True)
save_jsonl(test_set, f"{OUTPUT_DIR}/test.jsonl", strip_category=False)  # keep category for eval